# Generator-Aware Ridge Logistic Regression Sample

**Model:** L2-regularized logistic regression (binomial GLM) optimized with LBFGS. This is not a custom neural/network architecture; the main customization is the explicit generator-aware feature basis and the final logit transform.

**Features:** GPT-2 BPE income-token hierarchy, exact/integer commute keys, cross-fitted smoothed target-rate log-odds, token × context cells, commute × context cells, category × continuous-bin cells, generator-formula cells, category pairs, train+test frequency/composition features, raw numeric features, hinge features, and one-hot categorical features.

**Training:** 10-fold stratified CV. For every outer fold, label-derived features for outer-training rows are built with 5-fold inner cross-fitting; outer-validation and test rows use statistics from the complete outer-training fold. Outputs: consolidated OOF/TEST Parquets and `submission.csv`.

In [ ]:
from __future__ import annotations

import gc
import importlib.metadata
import json
import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

IS_KAGGLE = os.getenv('KAGGLE_KERNEL_RUN_TYPE') is not None
SMOKE_TEST = os.getenv('GENERATOR_LOGREG_SMOKE', '0') == '1'
DATA_DIR = (Path('/kaggle/input/competitions/playground-series-s6e9')
            if IS_KAGGLE else Path('data'))
ORIG_DIR = (Path('/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety')
            if IS_KAGGLE else Path('data'))
ORIGINAL_PATH = ORIG_DIR / ('EV_Adoption_and_Range_Anxiety_Dataset.csv'
                            if IS_KAGGLE else 'train_original.csv')
EXP_NAME = MODEL_NAME = 'GENERATOR_AWARE_REALMLP'
OOF_DIR, TEST_PRED_DIR = Path('oof'), Path('test_preds')
SUBMISSION_PATH = Path('submission.csv')
ARTIFACT_DIR = Path('artifacts_for_analysis') / EXP_NAME / 'files'

N_FOLDS, INNER_FOLDS, RANDOM_STATE = 10, 5, 42
L2 = 10.0
LBFGS_ITERS = 15 if SMOKE_TEST else 200
OUT_TEMPERATURE, OUT_OFFSET = 1.25, -1.0
SMOKE_TRAIN_ROWS, SMOKE_TEST_ROWS, SMOKE_FOLDS = 1200, 128, 2
TARGET, ID_COL = 'Will_Buy_EV', 'id'
FLOAT_COLS = ['Annual_Income_USD', 'Daily_Commute_km', 'Environmental_Concern_Level']
INT_COLS = ['Age', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work']
CAT_COLS = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
COLS = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
        'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
        'Environmental_Concern_Level'] + CAT_COLS

if SMOKE_TEST:
    OOF_DIR /= '_smoke'
    TEST_PRED_DIR /= '_smoke'
    SUBMISSION_PATH = Path('_smoke') / 'submission.csv'
    ARTIFACT_DIR = Path('artifacts_for_analysis') / (EXP_NAME + '_SMOKE') / 'files'
for d in (OOF_DIR, TEST_PRED_DIR, SUBMISSION_PATH.parent, ARTIFACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    total = torch.cuda.get_device_properties(0).total_memory
    torch.cuda.set_per_process_memory_fraction(min(1.0, 9.0 * 2**30 / total))
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
np.random.seed(RANDOM_STATE)
TIMINGS = []


def tick(stage, started, **extra):
    TIMINGS.append({'stage': stage, 'seconds': round(time.time() - started, 3), **extra})


def write_json(name, value):
    (ARTIFACT_DIR / name).write_text(json.dumps(value, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)))

print(f'{EXP_NAME} | {"SMOKE" if SMOKE_TEST else "CV"} | device={DEVICE}')

In [ ]:
t0 = time.time()
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

missing_train = set([ID_COL, TARGET] + COLS) - set(train.columns)
missing_test = set([ID_COL] + COLS) - set(test.columns)
if missing_train or missing_test:
    raise ValueError(f'missing columns: train={sorted(missing_train)} test={sorted(missing_test)}')
if TARGET in test.columns:
    raise ValueError('test.csv must not contain the target column')
if not train[ID_COL].is_unique or not test[ID_COL].is_unique:
    raise ValueError('train/test IDs must be unique')

ORIGINAL_TRAIN_ROW_INDEX = np.arange(len(train), dtype=np.int64)
if SMOKE_TEST:
    if len(train) > SMOKE_TRAIN_ROWS:
        idx, _ = train_test_split(
            np.arange(len(train)), train_size=SMOKE_TRAIN_ROWS,
            stratify=train[TARGET], random_state=RANDOM_STATE)
        idx = np.sort(idx)
        ORIGINAL_TRAIN_ROW_INDEX = ORIGINAL_TRAIN_ROW_INDEX[idx]
        train = train.iloc[idx].reset_index(drop=True)
    test = test.iloc[:SMOKE_TEST_ROWS].reset_index(drop=True)

y = (train[TARGET] == 'Yes').astype(np.int8).to_numpy()
ntr, nte = len(train), len(test)
X = pd.concat([train[COLS], test[COLS]], ignore_index=True)
N = len(X)
SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(np.zeros(ntr), y))
FOLD_ID = np.full(ntr, -1, np.int8)
for k, (_, va) in enumerate(SPLITS):
    FOLD_ID[va] = k

tick('load_data', t0, rows_train=ntr, rows_test=nte)
print('Train:', train.shape, '| test:', test.shape)

In [ ]:
t0 = time.time()
try:
    from transformers import AutoTokenizer
except ImportError as e:
    raise ImportError('This notebook requires the transformers package for the GPT-2 income-token features.') from e

TOK = AutoTokenizer.from_pretrained('gpt2')
inc_str = X.Annual_Income_USD.astype(np.int64).astype(str).to_numpy()
u_inc, inv_inc = np.unique(inc_str, return_inverse=True)
tok_ids = [TOK(' ' + s).input_ids for s in u_inc]


def fz(values):
    return pd.factorize(values, sort=False)[0].astype(np.int64)


def qbin(values, q):
    ranks = pd.Series(values).rank(method='first')
    return fz(pd.qcut(ranks, q, labels=False, duplicates='drop').to_numpy())

K = {}
K['L1'] = fz(np.array([t[0] for t in tok_ids], dtype=np.int64)[inv_inc])
K['L2'] = fz(np.array([t[0] * 60000 + (t[1] if len(t) > 1 else -1) for t in tok_ids], dtype=np.int64)[inv_inc])
K['IV'] = fz(inc_str)
K['LAST'] = fz(np.array([len(t) * 60000 + t[-1] for t in tok_ids], dtype=np.int64)[inv_inc])
cm = X.Daily_Commute_km.to_numpy(np.float64)
K['CV'] = fz(cm)
K['CI'] = fz(np.floor(cm))
for c in COLS:
    if c not in ('Annual_Income_USD', 'Daily_Commute_km'):
        K[c] = fz(X[c].to_numpy())
K['AGEDEC'] = fz((X.Age // 10).to_numpy())
K['CMTBIN'] = qbin(cm, 10)
K['SHB'] = fz(np.minimum(X.Charging_Stations_Near_Home, 6).to_numpy())
K['SWB'] = fz(np.minimum(X.Charging_Stations_Near_Work, 6).to_numpy())
K['INCQ'] = qbin(X.Annual_Income_USD.to_numpy(), 5)
K['INC20'] = qbin(X.Annual_Income_USD.to_numpy(), 20)
K['INC_MOD100'] = fz((X.Annual_Income_USD.astype(np.int64) % 100).to_numpy())
K['INC_DIV1000'] = fz((X.Annual_Income_USD.astype(np.int64) // 1000).to_numpy())
K['CELL'] = fz(((K['Subsidy_Available'] * 5 + K['Environmental_Concern_Level']) * 3 + K['Range_Anxiety_Level']) * 2 + K['Home_Charging_Possible'])


def cross(a, b):
    return fz(K[a] * (int(K[b].max()) + 1) + K[b])

FLAT, FAMILY = {}, {}
for lv in ['L1', 'L2', 'LAST', 'IV', 'CV', 'CI', 'INC_MOD100', 'INC_DIV1000', 'Age']:
    for a in (5, 50):
        name = f'r_{lv}_a{a}'
        FLAT[name] = (K[lv], a)
        FAMILY[name] = 'flat_rate'
for lv in ['L1', 'L2', 'LAST']:
    for c in ['City_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Environmental_Concern_Level',
              'Range_Anxiety_Level', 'Gender', 'Current_Car_Type', 'Number_of_Cars_Owned',
              'AGEDEC', 'CMTBIN', 'SHB', 'SWB']:
        name = f'tokx_{lv}_{c}'
        FLAT[name] = (cross(lv, c), 20)
        FAMILY[name] = 'token_x_context'
for c in ['City_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Environmental_Concern_Level', 'INCQ']:
    name = f'cmtx_CI_{c}'
    FLAT[name] = (cross('CI', c), 20)
    FAMILY[name] = 'commute_x_context'
for c in ['Home_Charging_Possible', 'City_Type', 'Range_Anxiety_Level']:
    name = f'cmtx_CV_{c}'
    FLAT[name] = (cross('CV', c), 20)
    FAMILY[name] = 'commute_x_context'
for name, key, alpha in [
    ('cell', K['CELL'], 5), ('cell_x_L1', cross('CELL', 'L1'), 20),
    ('cell_x_CI', cross('CELL', 'CI'), 20), ('cell_x_INCQ', cross('CELL', 'INCQ'), 20)]:
    FLAT[name] = (key, alpha)
    FAMILY[name] = 'generator_formula_cell'

CATS = ['Environmental_Concern_Level', 'Subsidy_Available', 'Range_Anxiety_Level', 'Home_Charging_Possible',
        'City_Type', 'Current_Car_Type', 'Number_of_Cars_Owned', 'Gender', 'AGEDEC', 'SHB', 'SWB']
for c in CATS:
    for suffix, other in [('INC20', 'INC20'), ('CMT10', 'CMTBIN')]:
        name = f'xc_{c}_{suffix}'
        FLAT[name] = (cross(c, other), 20)
        FAMILY[name] = 'category_x_continuous_bin'
for i, c1 in enumerate(CATS):
    for c2 in CATS[i + 1:]:
        name = f'xp_{c1}_{c2}'
        FLAT[name] = (cross(c1, c2), 20)
        FAMILY[name] = 'category_pair'
K['CELLI'] = cross('CELL', 'INC20')
for name, key in [('x3_CELL_INC20', K['CELLI']), ('x3_CELL_CMT10', cross('CELL', 'CMTBIN'))]:
    FLAT[name] = (key, 20)
    FAMILY[name] = 'cell_x_bin'
for c in ['City_Type', 'Current_Car_Type', 'Number_of_Cars_Owned', 'AGEDEC']:
    name = f'x4_CELLI_{c}'
    FLAT[name] = (cross('CELLI', c), 20)
    FAMILY[name] = 'cell_x_bin'

CHAINS = {f'chain_S{S}': ([K['L1'], K['L2'], K['IV']], S) for S in (5, 20, 80)}
CHAINS.update({f'chainC_S{S}': ([K['CI'], K['CV']], S) for S in (5, 20)})
LABEL_NAMES = list(FLAT) + [f'{n}_lv{i}' for n, (ks, _) in CHAINS.items() for i in range(len(ks))]
for n, (ks, _) in CHAINS.items():
    for i in range(len(ks)):
        FAMILY[f'{n}_lv{i}'] = 'token_tree_beta_chain' if n.startswith('chain_') else 'commute_beta_chain'

LF, LFN = [], []
for nm in ['IV', 'L2', 'L1', 'CV', 'LAST']:
    LF.append(np.log1p(np.bincount(K[nm])[K[nm]]))
    LFN.append(f'cnt_{nm}')
    FAMILY[f'cnt_{nm}'] = 'count_train_test'
inc_f = X.Annual_Income_USD.to_numpy(np.float64)
env = X.Environmental_Concern_Level.to_numpy(np.float64)
subs = (X.Subsidy_Available == 'Yes').to_numpy()
anx_formula = X.Range_Anxiety_Level.map({'Low': 0, 'Medium': 1, 'High': 3}).to_numpy(np.float64)
LF.append(1.2 * inc_f / 1e5 + 0.6 * env + 2 * subs - anx_formula)
LFN.append('formula'); FAMILY['formula'] = 'generator_formula'
for c in COLS[:7]:
    v = X[c].to_numpy(np.float64)
    LF.append(v); LFN.append(f'raw_{c}'); FAMILY[f'raw_{c}'] = 'raw'
    for q in (0.1, 0.3, 0.5, 0.7, 0.9):
        knot = np.quantile(v, q)
        name = f'hinge_{c}_{q}'
        LF.append(np.maximum(v - knot, 0.0)); LFN.append(name); FAMILY[name] = 'raw_hinge'
OH = pd.get_dummies(X[CAT_COLS + ['Environmental_Concern_Level', 'Number_of_Cars_Owned']].astype(str), dtype=np.float32)
for c in OH.columns:
    name = f'oh_{c}'; LFN.append(name); FAMILY[name] = 'one_hot'
LF = np.column_stack(LF + [OH.to_numpy()]).astype(np.float32)

MIXV = {
    'env': env, 'sub': subs.astype(float),
    'anx': X.Range_Anxiety_Level.map({'Low': 0, 'Medium': 1, 'High': 2}).to_numpy(np.float64),
    'home': (X.Home_Charging_Possible == 'Yes').to_numpy(float),
    'urban': (X.City_Type == 'Urban').to_numpy(float), 'rural': (X.City_Type == 'Rural').to_numpy(float),
    'suv': (X.Current_Car_Type == 'SUV').to_numpy(float), 'truck': (X.Current_Car_Type == 'Truck').to_numpy(float),
    'sedan': (X.Current_Car_Type == 'Sedan').to_numpy(float), 'male': (X.Gender == 'Male').to_numpy(float),
    'cars': X.Number_of_Cars_Owned.to_numpy(float), 'age': X.Age.to_numpy(float),
    'sth': X.Charging_Stations_Near_Home.to_numpy(float), 'stw': X.Charging_Stations_Near_Work.to_numpy(float),
    'cmt': cm, 'inc': inc_f / 1e4,
}
ivcv = fz(np.char.add(np.char.add(inc_str.astype(str), '_'), X.Daily_Commute_km.astype(str).to_numpy()))
mix_keys = {'IV': K['IV'], 'L2': K['L2'], 'L1': K['L1'], 'CV': K['CV'], 'IVCV': ivcv}
MIX, MIXN = [], []
for kn, key in mix_keys.items():
    counts = np.bincount(key).astype(np.float64)
    n = counts[key]
    MIX.append(np.log(np.maximum(n, 1.0))); MIXN.append(f'mix_{kn}_logn')
    for vn, v in MIXV.items():
        if (kn in ('IV', 'L2', 'L1') and vn == 'inc') or (kn == 'CV' and vn == 'cmt') or (kn == 'IVCV' and vn in ('inc', 'cmt')):
            continue
        global_mean = float(v.mean())
        sums = np.bincount(key, weights=v, minlength=len(counts))[key]
        MIX.append((sums - v + 5 * global_mean) / (n - 1 + 5) - global_mean)
        MIXN.append(f'mix_{kn}_{vn}')
for name in MIXN:
    FAMILY[name] = 'composition_mix_transductive'
MIX = np.column_stack(MIX).astype(np.float32)
LF = np.concatenate([LF, MIX], axis=1)
LFN += MIXN
ALL_NAMES = LABEL_NAMES + LFN
P = len(ALL_NAMES)

tick('feature_keys_and_label_free', t0, n_features=P)
print(f'Features: {P} | label-derived={len(LABEL_NAMES)} | label-free={len(LFN)}')

In [ ]:
def _stats(key, fit):
    size = int(key.max()) + 1
    return (np.bincount(key[fit], weights=y[fit], minlength=size),
            np.bincount(key[fit], minlength=size))


def _logit_rate(rate):
    rate = np.clip(rate, 1e-6, 1 - 1e-6)
    return np.log(rate / (1 - rate)).astype(np.float32)


def encode(fit, app, out, rows):
    prior = float(y[fit].mean())
    j = 0
    for _, (key, alpha) in FLAT.items():
        s, c = _stats(key, fit)
        ka = key[app]
        out[rows, j] = _logit_rate((s[ka] + alpha * prior) / (c[ka] + alpha))
        j += 1
    for _, (keys, strength) in CHAINS.items():
        post = np.full(len(app), prior, np.float64)
        for key in keys:
            s, c = _stats(key, fit)
            ka = key[app]
            post = (s[ka] + strength * post) / (c[ka] + strength)
            out[rows, j] = _logit_rate(post)
            j += 1


def build_fold(k):
    fit_rows, valid_rows = SPLITS[k]
    test_rows = np.arange(ntr, N)
    Mfit = np.empty((len(fit_rows), P), np.float32)
    Mvalid = np.empty((len(valid_rows), P), np.float32)
    Mtest = np.empty((nte, P), np.float32)

    n_label = len(LABEL_NAMES)
    encode(fit_rows, valid_rows, Mvalid, slice(None))
    encode(fit_rows, test_rows, Mtest, slice(None))

    position = np.full(ntr, -1, np.int64)
    position[fit_rows] = np.arange(len(fit_rows))
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for inner_fit_rel, inner_app_rel in inner.split(np.zeros(len(fit_rows)), y[fit_rows]):
        inner_fit = fit_rows[inner_fit_rel]
        inner_app = fit_rows[inner_app_rel]
        encode(inner_fit, inner_app, Mfit, position[inner_app])

    Mfit[:, n_label:] = LF[fit_rows]
    Mvalid[:, n_label:] = LF[valid_rows]
    Mtest[:, n_label:] = LF[ntr:]
    return fit_rows, valid_rows, Mfit, Mvalid, Mtest


def fit_glm(Z, yy, l2=L2, iters=LBFGS_ITERS):
    w = torch.zeros(Z.shape[1], device=Z.device, dtype=torch.float64, requires_grad=True)
    b = torch.zeros(1, device=Z.device, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([w, b], lr=1.0, max_iter=iters, history_size=20,
                            tolerance_grad=1e-9, tolerance_change=1e-12,
                            line_search_fn='strong_wolfe')

    def closure():
        opt.zero_grad()
        logits = Z @ w + b
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, yy, reduction='sum')
        loss = loss + 0.5 * l2 * (w * w).sum()
        loss.backward()
        return loss

    loss = float(opt.step(closure))
    return w.detach(), b.detach(), loss


# pvep: public/pub_realmlp.py'den RealMLP (yekenot) sınıfları ve ayarları
import math, random
import torch.nn as nn
import torch.nn.functional as F
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
SEED0 = RANDOM_STATE
# ── Preprocessing ─────────────────────────────────────────────────────────────
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    """
    Applies a configurable sequence of numerical transforms from CONFIG["tfms"].
    Supported: 'median_center', 'robust_scale', 'smooth_clip', 'l2_normalize'.
    'one_hot' and 'embedding' are recognised but skipped (handled by the model).
    """

    def __init__(self, tfms):
        self._tfms = [t for t in tfms
                      if t in ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]

    def fit(self, X: np.ndarray, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5 * (X.max(axis=0)[zero_idx] - X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0 / (q_diff + 1e-30)
            self._iqr_factors[q_diff == 0.0] = 0.0
        return self

    def transform(self, X: np.ndarray, y=None) -> np.ndarray:
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm == "median_center":
                X -= self._median[None, :]
            elif tfm == "robust_scale":
                X *= self._iqr_factors[None, :]
            elif tfm == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif tfm == "l2_normalize":
                norms = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(norms == 0, 1.0, norms)
        return X

# ── Model components ──────────────────────────────────────────────────────────
class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens: int, cat_dims, embed_dim: int = 8,
                 onehot_thresh: int = 8, device=None):
        super().__init__()
        self.n_ens = n_ens
        self.cat_dims = cat_dims
        self.onehot_features = []
        self.embed_layers = nn.ModuleList()
        self._embed_feature_indices = []

        for i, dim in enumerate(cat_dims):
            if dim <= onehot_thresh:
                self.onehot_features.append(i)
            else:
                emb = nn.ModuleList(
                    [nn.Embedding(dim, embed_dim) for _ in range(n_ens)]
                )
                self.embed_layers.append(emb)
                self._embed_feature_indices.append(i)

    def forward(self, x):
        # x: (batch, n_ens, n_cat)
        batch_size, n_ens, _ = x.shape
        features = []

        if self.onehot_features:
            onehot_x    = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_oh    = sum(onehot_dims)
            encoded     = torch.zeros(batch_size, n_ens, total_oh, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx : idx + 1].long()
                encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(encoded)

        for emb_list, feat_idx in zip(self.embed_layers, self._embed_feature_indices):
            feat_embs = []
            for model_idx in range(self.n_ens):
                indices = x[:, model_idx, feat_idx : feat_idx + 1].long()
                feat_embs.append(emb_list[model_idx](indices))    # (batch, 1, embed_dim)
            feat_combined = torch.cat(feat_embs, dim=1)           # (batch, n_ens, embed_dim)
            features.append(feat_combined)

        return torch.cat(features, dim=2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens: int, n_features: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]


class NTPLinear(nn.Module):
    def __init__(self, n_ens: int, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias   = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        # x: (batch, n_ens, in_features)
        # Single einsum replaces transpose → matmul → transpose
        x = torch.einsum("bki,kio->bko", x, self.weight) / math.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x


class ResidualBlock(nn.Module):
    def __init__(self, n_ens: int, dim: int, dropout: float, activation=nn.SiLU):
        super().__init__()
        self.linear = NTPLinear(
            n_ens=n_ens,
            in_features=dim,
            out_features=dim,
        )
        self.act = activation()
        self.drop = nn.Dropout(dropout)
        # Learnable residual strength
        self.res_scale = nn.Parameter(torch.ones(n_ens, dim) * 0.1)

    def forward(self, x):
        residual = x
        x = self.linear(x)
        x = self.act(x)
        x = self.drop(x)
        return residual + x * self.res_scale.unsqueeze(0)


class PBLDEmbedding(nn.Module):
    """Periodic Basis with Learned Decay embedding for numerical features."""

    def __init__(self, n_ens: int, n_features: int,
                 hidden_dim: int = 16, out_dim: int = 4, freq_scale: float = 0.1,
                 activation=nn.GELU):
        super().__init__()
        self.n_ens      = n_ens
        self.n_features = n_features
        self.out_dim    = out_dim
        self.w1 = nn.Parameter(torch.empty(n_ens, n_features, hidden_dim))
        nn.init.normal_(self.w1, mean=0.0, std=freq_scale / math.sqrt(hidden_dim))
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(
            torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) / math.sqrt(hidden_dim)
        )
        self.b2 = nn.Parameter(torch.zeros(n_ens, n_features, out_dim - 1))
        self.act = activation()
        nn.init.uniform_(self.b1, -math.pi, math.pi)

    def forward(self, x):
        # x: (batch, n_ens, n_features)
        # All operations are fully vectorised over n_features — no Python loop.

        # (batch, n_ens, n_features, 1) * (n_ens, n_features, hidden)
        # → periodic: (batch, n_ens, n_features, hidden)
        periodic = torch.cos(
            2 * math.pi * (
                x.unsqueeze(-1) * self.w1.unsqueeze(0)   # Broadcast over batch
                + self.b1.unsqueeze(0)
            )
        )

        # (batch, n_ens, n_features, hidden) @ (n_ens, n_features, hidden, out-1)
        # → transformed: (batch, n_ens, n_features, out-1)
        transformed = self.act(
            torch.einsum("bkfh,kfhd->bkfd", periodic, self.w2)
            + self.b2.unsqueeze(0)
        )

        # Concatenate raw feature (residual) with transformed output
        # x: (batch, n_ens, n_features) → unsqueeze → (batch, n_ens, n_features, 1)
        feat = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        # (batch, n_ens, n_features, out_dim) → flatten last two dims
        # → (batch, n_ens, n_features * out_dim)
        return feat.flatten(start_dim=2)

# ── Model ─────────────────────────────────────────────────────────────
class RealMLP(nn.Module):
    def __init__(self, output_dim: int, cat_dims, n_numerical: int, cfg: dict):
        super().__init__()
        n_ens      = cfg["n_ens"]
        embed_dim  = cfg["embed_dim"]
        self.n_ens = n_ens

        self.cate = CategoricalFeatureLayer(
            n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim,
            onehot_thresh=cfg["onehot_thresh"],
        )
        self.num_embed = PBLDEmbedding(
            n_ens=n_ens,
            n_features=n_numerical,
            hidden_dim=cfg["pbld_hidden_dim"],
            out_dim=cfg["pbld_out_dim"],
            freq_scale=cfg["pbld_freq_scale"],
            activation=cfg["pbld_activation"],
        )

        num_emb_dim = n_numerical * cfg["pbld_out_dim"]
        cat_emb_dim = sum(
            c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims
        )
        total_dim = num_emb_dim + cat_emb_dim
        hidden_dims = cfg["hidden_dims"]

        act = cfg["activation"]

        # Build layers, tracking which NTPLinear is the "first layer"
        # so we can give it a separate lr group.
        # Each hidden position gets its own Dropout instance (shared instance
        # would only register once in nn.Sequential and break the scheduler).
        self._dropout_modules = []    # Kept for live p_drop_sched updates

        layers = []
        if cfg["add_front_scale"]:
            layers.append(ScalingLayer(n_ens=n_ens, n_features=total_dim))

        in_dim = total_dim
        # First projection layer
        first_linear = NTPLinear(
            n_ens=n_ens,
            in_features=in_dim,
            out_features=hidden_dims[0],
        )
        self.first_linear = first_linear
        layers.extend([first_linear, act()])

        in_dim = hidden_dims[0]
        # Residual blocks
        for hdim in hidden_dims[1:]:
            # Transition layer if dimensions differ
            if in_dim != hdim:
                layers.extend([
                    NTPLinear(
                        n_ens=n_ens,
                        in_features=in_dim,
                        out_features=hdim,
                    ),
                    act(),
                ])
                in_dim = hdim

            block = ResidualBlock(
                n_ens=n_ens,
                dim=hdim,
                dropout=cfg["dropout"],
                activation=act,
            )

            self._dropout_modules.append(block.drop)
            layers.append(block)

        self.hidden = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=output_dim)

        with torch.no_grad():
            self.output_layer.weight.mul_(0.1)
            if self.output_layer.bias is not None:
                self.output_layer.bias.zero_()

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        combined = torch.cat([x_num, x_cat], dim=2)
        x = self.hidden(combined)
        x = self.output_layer(x)
        return x    # (batch, n_ens, 1)

# ── Schedule helpers ─────────────────────────────────────────────────────────
def apply_schedule(init_value: float, progress: float, sched: str,
                   flat_ratio: float = 0.3) -> float:
    """
    Supported schedules:
      'constant'    – no decay
      'cos'         – cosine from init to 0
      'flat_cos'    – flat for flat_ratio, then cosine to 0
      'flat_anneal' – flat for flat_ratio, then linear to 0
      'sqrt_cos'    – sqrt of cosine annealing (slower decay)
      'expm4t'      – exponential decay: init * exp(-4 * progress)
    """
    if sched == "constant":
        return init_value
    elif sched == "cos":
        return init_value * (math.cos(math.pi * progress) + 1) / 2
    elif sched == "flat_cos":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (math.cos(math.pi * t) + 1) / 2
    elif sched == "flat_anneal":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (1 - t)
    elif sched == "sqrt_cos":
        return init_value * math.sqrt((math.cos(math.pi * progress) + 1) / 2)
    elif sched == "expm4t":
        return init_value * math.exp(-4 * progress)
    else:
        raise ValueError(f"Unknown schedule: '{sched}'")

# ── Per-parameter-group builder ───────────────────────────────────────────────
def get_parameter_groups(model: RealMLP, p: dict):
    """
    Five groups with independent lr / wd:
      0 – ScalingLayer params  (scale.*)
      1 – PBLD / num_embed params
      2 – first hidden linear weight
      3 – all other weights
      4 – all biases (excluding those already in groups 0-2)
    Note: PBLD has its own bias params (b1, b2) which belong to group 1, not 4.
    The ordering of checks therefore is: scale → num_embed → first_w → bias → other_w.
    """
    first_linear_weight_id = id(model.first_linear.weight)

    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if "num_embed" in name:
            # All PBLD params (weights and biases) get their own lr group
            pbld_p.append(param)
        elif "scale" in name:
            scale_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif "bias" in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)

    LR = p["lr"]
    WD = p["weight_decay"]
    return [
        {"params": scale_p,   "lr": LR * p["lr_scale_mult"],         "weight_decay":  WD * p["wd_scale_mult"],         "group": "scale"},
        {"params": pbld_p,    "lr": LR * p["pbld_lr_factor"],        "weight_decay":  WD,                              "group": "pbld"},
        {"params": first_w_p, "lr": LR * p["first_layer_lr_factor"], "weight_decay":  WD * p["first_layer_wd_factor"], "group": "first_w"},
        {"params": other_w_p, "lr": LR,                              "weight_decay":  WD,                              "group": "other_w"},
        {"params": bias_p,    "lr": LR * p["lr_bias_mult"],          "weight_decay":  WD * p["wd_bias_mult"],          "group": "bias"},
    ]

# ── Binary Cross-Entropy Loss with optional class weights ───────
def binary_bce_loss(
    y_true: torch.Tensor,
    logits: torch.Tensor,
    ls: float = 0.0,
    pos_weight: torch.Tensor = None,
) -> torch.Tensor:
    """
    y_true : (N,) float {0,1}
    y_pred : (N,) sigmoid probabilities
    """

    # Binary label smoothing
    if ls > 0.0:
        y_true = y_true * (1.0 - ls) + 0.5 * ls

    if pos_weight is None:
        loss = (
            (1.0 - y_true) * logits
            + F.softplus(-logits)
        )
    else:
        loss = (
            (1.0 - y_true) * logits
            + (1.0 + (pos_weight - 1.0) * y_true)
            * F.softplus(-logits)
        )

    return loss.mean()

# ── Sklearn-compatible wrapper ────────────────────────────────────────────────
class RealMLP_TD_Classifier(BaseEstimator):
    """
    Sklearn-compatible wrapper around RealMLP, matching the interface of
    pytabkit's RealMLP_TD_Classifier:
    model = RealMLP_TD_Classifier(**CONFIG)
    model.fit(X_train, y_train, X_val, y_val, cat_col_names=CATS)
    proba = model.predict_proba(X_test)
    """

    def __init__(self, **kwargs):
        # Accept any subset of CONFIG keys; fall back to CONFIG defaults
        self.params = {**CONFIG, **kwargs}

    def fit(self, X_train: pd.DataFrame, y_train, X_val: pd.DataFrame, y_val,
            cat_col_names=None, X_test: pd.DataFrame = None):
        p   = self.params
        dev = torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        verbose = p["verbosity"]
        cat_col_names = cat_col_names or []
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]

        # ── Split num / cat ──────────────────────────────────────────────────
        X_tr_num  = X_train[num_col_names].values.astype(np.float32)
        X_val_num = X_val[num_col_names].values.astype(np.float32)
        X_tr_cat  = X_train[cat_col_names].values.astype(np.int64)
        X_val_cat = X_val[cat_col_names].values.astype(np.int64)
        y_tr      = np.asarray(y_train)
        y_v       = np.asarray(y_val)

        # ── Numerical preprocessing ───────────────────────────────────────────
        self.preprocessor_ = NumericalPreprocessor(p["tfms"])
        self.preprocessor_.fit(X_tr_num)
        X_tr_num  = self.preprocessor_.transform(X_tr_num)
        X_val_num = self.preprocessor_.transform(X_val_num)

        # ── Categorical dims ─────────────────────────────────────────────────────────
        self.cat_col_names_ = cat_col_names
        self.num_col_names_ = num_col_names
        if cat_col_names:
            all_cat = [X_tr_cat, X_val_cat]
            if X_test is not None:
                all_cat.append(X_test[cat_col_names].values.astype(np.int64))
            cat_dims = (np.concatenate(all_cat, axis=0).max(axis=0) + 1).tolist()
        else:
            cat_dims = []
        self.cat_dims_ = cat_dims

        # Clamp indices to [0, dim-1] — -1 codes (unseen pandas categories)
        # wrap to huge ints on GPU and cause device-side assert
        if cat_dims:
            cat_max = np.array(cat_dims) - 1
            X_tr_cat  = np.clip(X_tr_cat,  0, cat_max)
            X_val_cat = np.clip(X_val_cat, 0, cat_max)

        # ── Class weights ────────────────────────────────────────────────────
        classes       = np.unique(y_tr)
        self.classes_ = classes
        weights_np = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_tr,
        )

        # Sklearn order follows sorted(classes)
        # Class 1 weight becomes BCE positive weight
        pos_weight = torch.tensor(weights_np[1], dtype=torch.float32, device=dev)

        # ── Build model ──────────────────────────────────────────────────────
        self.model_ = RealMLP(
            output_dim=1,
            cat_dims=cat_dims,
            n_numerical=X_tr_num.shape[1],
            cfg=p,
        ).to(dev)

        param_groups = get_parameter_groups(self.model_, p)
        # Store the base lr on each group so the scheduler can scale from it
        for g in param_groups:
            g["lr_base"] = g["lr"]
        optimizer = torch.optim.AdamW(
            param_groups,
            betas=(p["mom"], p["sq_mom"]),
        )

        # ── To tensors ───────────────────────────────────────────────────────
        Xtn = torch.as_tensor(X_tr_num,  dtype=torch.float32, device=dev)
        Xtc = torch.as_tensor(X_tr_cat,  dtype=torch.long,    device=dev)
        ytt = torch.as_tensor(y_tr,      dtype=torch.float32, device=dev)
        Xvn = torch.as_tensor(X_val_num, dtype=torch.float32, device=dev)
        Xvc = torch.as_tensor(X_val_cat, dtype=torch.long,    device=dev)

        n_ens       = p["n_ens"]
        train_bs    = p["train_bs"]
        eval_bs     = p["eval_bs"]
        epochs      = p["epochs"]
        lr_sched    = p["lr_sched"]
        flat_ratio  = p["flat_ratio"]
        ema_decay   = p["ema_decay"]
        total_steps = epochs * len(y_tr)
        train_order = np.arange(len(y_tr))

        best_score      = -np.inf
        best_epoch      = 0
        best_val_probs  = None
        best_state      = None
        ema_state       = None
        if ema_decay > 0:
            ema_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}

        # ── Epoch loop ───────────────────────────────────────────────────────
        for epoch in range(epochs):
            self.model_.train()
            for start in range(0, len(y_tr), train_bs):
                progress  = (epoch * len(y_tr) + start) / total_steps
                idx_batch = train_order[start : start + train_bs]

                # Update lr for each param group using its base lr
                for g in optimizer.param_groups:
                    g["lr"] = apply_schedule(g["lr_base"], progress, lr_sched, flat_ratio)

                optimizer.zero_grad()
                y_pred = self.model_(Xtn[idx_batch], Xtc[idx_batch])    # (bs, n_ens, C)

                ls_val   = apply_schedule(p["ls_eps"],  progress, p["ls_eps_sched"],  flat_ratio)
                drop_val = apply_schedule(p["dropout"], progress, p["p_drop_sched"],  flat_ratio)
                for dm in self.model_._dropout_modules:
                    dm.p = drop_val

                loss = binary_bce_loss(
                    ytt[idx_batch].repeat_interleave(n_ens),
                    y_pred.reshape(-1),
                    ls=ls_val,
                    pos_weight=None,
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), p["grad_clip"])
                optimizer.step()

                if ema_state is not None:
                    with torch.no_grad():
                        model_state = self.model_.state_dict()
                        for key, value in model_state.items():
                            if torch.is_floating_point(value):
                                ema_state[key].mul_(ema_decay).add_(value.detach(), alpha=1.0 - ema_decay)
                            else:
                                ema_state[key].copy_(value)

            np.random.shuffle(train_order)

            # ── Validation ───────────────────────────────────────────────────
            self.model_.eval()

            live_state = None
            if ema_state is not None:
                live_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}
                self.model_.load_state_dict(ema_state, strict=True)
            
            with torch.no_grad():
                val_probs_pos = np.concatenate([
                    torch.sigmoid(self.model_(Xvn[s : s + eval_bs], Xvc[s : s + eval_bs]))
                        .mean(dim=1)
                        .squeeze(-1)
                        .cpu()
                        .numpy()
                    for s in range(0, len(y_v), eval_bs)
                ], axis=0)

                val_probs = np.stack(
                    [1.0 - val_probs_pos, val_probs_pos],
                    axis=1,
                )

            val_pred = val_probs[:, 1]
            epoch_score = roc_auc_score(y_v, val_pred)
            improved    = epoch_score > best_score
            if improved:
                best_score     = epoch_score
                best_epoch     = epoch + 1
                best_val_probs = val_probs.copy()
                state_src      = ema_state if ema_state is not None else self.model_.state_dict()
                best_state     = {k: v.detach().clone() for k, v in state_src.items()}

            # Restore non-EMA/live weights before continuing training
            if live_state is not None:
                self.model_.load_state_dict(live_state, strict=True)

            if verbose >= 2:
                print(
                    f"  epoch {epoch + 1}/{epochs}  "
                    f"score = {epoch_score:.5f}  "
                    f"best = {best_score:.5f}  "
                    f"ls = {ls_val:.4f}  drop = {drop_val:.4f}"
                    + (" ✓" if improved else "")
                )

            # ── Early stopping ────────────────────────────────────────────────
            if p["use_early_stopping"]:
                patience = (best_epoch * p["early_stopping_multiplicative_patience"]
                            + p["early_stopping_additive_patience"])
                if (epoch + 1) > patience:
                    if verbose >= 1:
                        print(f"  Early stopping at epoch {epoch + 1} "
                              f"(best epoch {best_epoch})")
                    break

        # ── Restore best weights ──────────────────────────────────────────────
        if best_state is not None:
            self.model_.load_state_dict(best_state, strict=True)
            
        self.best_score_     = best_score
        self.best_val_probs_ = best_val_probs
        self._dev            = dev
        if verbose >= 1:
            print(f"  → best score: {best_score:.5f}  (epoch {best_epoch})")
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        eval_bs = self.params["eval_bs"]
        X_num = self.preprocessor_.transform(
            X[self.num_col_names_].values.astype(np.float32)
        )
        X_cat = X[self.cat_col_names_].values.astype(np.int64)
        # Clamp to valid embedding range — guards against -1 codes (unseen
        # categories in pandas category dtype) which wrap to large ints on GPU
        X_cat = np.clip(X_cat, 0, np.array(self.cat_dims_) - 1)
        Xn = torch.as_tensor(X_num, dtype=torch.float32, device=self._dev)
        Xc = torch.as_tensor(X_cat, dtype=torch.long,    device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            probs_pos = np.concatenate([
                torch.sigmoid(self.model_(Xn[s : s + eval_bs], Xc[s : s + eval_bs]))
                    .mean(dim=1)
                    .squeeze(-1)
                    .cpu()
                    .numpy()
                for s in range(0, len(X_num), eval_bs)
            ], axis=0)

        return np.stack(
            [1.0 - probs_pos, probs_pos],
            axis=1,
        )

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        probs = self.predict_proba(X)[:, 1]
        return self.classes_[(probs >= 0.5).astype(np.int64)]
# ----- cell
CONFIG = {
    # --- Model architecture ---
    "n_ens":         8,                # Number of ensemble members inside one RealMLP
    "embed_dim":     6,                # Embedding dim for high-cardinality categoricals
    "onehot_thresh": 4,                # Categoricals with nunique <= this get one-hot encoded
    "hidden_dims":   [256, 256, 256],  # MLP hidden layer sizes
    "dropout":       0.05,             # Base dropout probability value
    "p_drop_sched":  "expm4t",         # 'expm4t' | 'flat_cos' | 'constant'
    "activation":    nn.SiLU,          # Activation class for MLP layers
    "add_front_scale": True,           # Whether to prepend a ScalingLayer before MLP layers

    # --- PBLD (periodic) embedding for numericals ---
    "pbld_hidden_dim": 20,             # plr_hidden_1 in official pytabkit's API
    "pbld_out_dim":    5,              # plr_hidden_2 + 1 (includes residual raw feature)
    "pbld_freq_scale": 5.0,            # plr_sigma: init std of frequency weights
    "pbld_activation": nn.PReLU,       # plr_act_name: activation inside PBLD
    "pbld_lr_factor":  0.093,          # plr_lr_factor: PBLD param lr = lr * pbld_lr_factor

    # --- Optimizer ---
    "lr":               0.01,          # Base learning rate (weights)
    "mom":              0.9,           # Adam beta1
    "sq_mom":           0.99,          # Adam beta2
    "lr_sched":        "flat_anneal",  # 'flat_cos' | 'flat_anneal' | 'cos' | 'constant'
    "flat_ratio":       0.3,           # Flat fraction for flat_cos / flat_anneal schedules
    "first_layer_lr_factor": 1.2,      # lr multiplier for first MLP layer weights
    "first_layer_wd_factor": 0.1,      # wd multiplier for first MLP layer weights
    "lr_scale_mult":    10.0,          # Scale-layer lr = lr * lr_scale_mult
    "lr_bias_mult":     0.1,           # Bias        lr = lr * lr_bias_mult
    "weight_decay":     0.013,         # Base weight decay (weights)
    "wd_scale_mult":    0.1,           # Scale-layer wd = weight_decay * wd_scale_mult
    "wd_bias_mult":     0.5,           # Bias        wd = weight_decay * wd_bias_mult
    "ema_decay":        0.997875,      # Exponential Moving Average decay of model weights
    "grad_clip":        1.0,           # Graient clipping value

    # --- Label smoothing ---
    "ls_eps":       0.04,   # Base label smoothing epsilon
    "ls_eps_sched": "cos",  # 'cos' | 'sqrt_cos' | 'constant'

    # --- Preprocessing ---
    # Supported tfms (applied in order to numerical features):
    # 'median_center', 'robust_scale', 'smooth_clip', 'l2_normalize'
    # Categorical tfms ('one_hot', 'embedding') are handled separately by the model
    "tfms": ["median_center", "robust_scale", 'smooth_clip'],

    # --- Training loop ---
    "epochs":    2,
    "train_bs":  256,
    "eval_bs":   10240,
    "verbosity": 2,      # 0 = silent, 1 = fold summary only, 2 = per-epoch

    # --- Early stopping ---
    "use_early_stopping":                     False,
    "early_stopping_additive_patience":       10,     # Stop if epoch > best * mult + add
    "early_stopping_multiplicative_patience": 1,

    # --- Device ---
    "device":       "cuda",
    "random_state": SEED0,
}


def to_probability(eta):
    z = np.clip(OUT_TEMPERATURE * eta + OUT_OFFSET, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
run_started = time.time()
run_folds = list(range(SMOKE_FOLDS)) if SMOKE_TEST else list(range(N_FOLDS))
oof = np.full(ntr, np.nan, np.float64)
test_fold_predictions = []
fold_auc = []
coef_rows = []
summary = {'experiment': EXP_NAME, 'status': 'running', 'smoke_test': SMOKE_TEST, 'completed_folds': 0, 'fold_auc': []}
write_json('run_summary.json', summary)

for k in run_folds:
    fold_started = time.time()
    fit_rows, valid_rows, Mfit, Mvalid, Mtest = build_fold(k)
    if not (np.isfinite(Mfit).all() and np.isfinite(Mvalid).all() and np.isfinite(Mtest).all()):
        raise ValueError(f'non-finite feature detected in fold {k}')

    cols = [f'_f{i}' for i in range(P)]  # pvep: GLR özellik matrisi RealMLP'ye (hepsi sayısal)
    dfit, dval, dtst = (pd.DataFrame(M, columns=cols) for M in (Mfit, Mvalid, Mtest))
    for d_ in (dfit, dval, dtst):
        d_['dummy_'] = 0  # kategorik katman boş kalmasın diye sabit sütun
    model = RealMLP_TD_Classifier(**CONFIG)
    model.fit(dfit, pd.Series(y[fit_rows]), dval, pd.Series(y[valid_rows]), cat_col_names=['dummy_'], X_test=dtst)
    valid_pred = model.best_val_probs_[:, 1]
    test_pred = model.predict_proba(dtst)[:, 1]
    del dfit, dval, dtst, model
    oof[valid_rows] = valid_pred
    test_fold_predictions.append(test_pred)
    auc = float(roc_auc_score(y[valid_rows], valid_pred))
    fold_auc.append(auc)

    summary.update(completed_folds=len(fold_auc), fold_auc=fold_auc)
    write_json('run_summary.json', summary)
    print(f'FOLD {k + 1}/{N_FOLDS} | AUC={auc:.9f} | seconds={time.time() - fold_started:.1f}')

    del Mfit, Mvalid, Mtest
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

complete = bool(np.isfinite(oof).all())
test_pred = np.mean(np.stack(test_fold_predictions), axis=0)
if not np.isfinite(test_pred).all() or np.any((test_pred < 0) | (test_pred > 1)):
    raise ValueError('invalid test probabilities')

if complete:
    pooled_auc = float(roc_auc_score(y, oof))
    print(f'POOLED_OOF_AUC={pooled_auc:.12f}')
else:
    pooled_auc = None
    print('SMOKE run: partial OOF only; pooled CV AUC is not reported.')

In [ ]:
oof_frame = pd.DataFrame({
    ID_COL: train[ID_COL].to_numpy(),
    'row_index': ORIGINAL_TRAIN_ROW_INDEX,
    'fold': FOLD_ID,
    'y_true': y,
    'oof_pred': oof,
})
test_frame = pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), 'test_pred': test_pred})
submission = pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), TARGET: test_pred})

oof_path = OOF_DIR / f'{MODEL_NAME}_OOF.parquet'
test_path = TEST_PRED_DIR / f'{MODEL_NAME}_TEST.parquet'
if complete:
    oof_frame.to_parquet(oof_path, index=False)
else:
    partial_path = OOF_DIR / f'{MODEL_NAME}_PARTIAL_OOF.parquet'
    oof_frame.loc[np.isfinite(oof_frame.oof_pred)].to_parquet(partial_path, index=False)
    oof_path = partial_path
test_frame.to_parquet(test_path, index=False)
submission.to_csv(SUBMISSION_PATH, index=False)

pd.DataFrame(coef_rows).to_csv(ARTIFACT_DIR / 'feature_importance.csv', index=False)
pd.DataFrame(TIMINGS).to_csv(ARTIFACT_DIR / 'timings.csv', index=False)
pd.DataFrame({
    'feature': ALL_NAMES,
    'family': [FAMILY[n] for n in ALL_NAMES],
    'fit_scope': [
        ('label-free train+test transductive' if FAMILY[n] in ('count_train_test', 'composition_mix_transductive')
         else 'label-free row-local' if FAMILY[n] in ('raw', 'raw_hinge', 'one_hot', 'generator_formula')
         else 'outer-train labels; outer-training rows inner 5-fold cross-fitted')
        for n in ALL_NAMES]
}).to_csv(ARTIFACT_DIR / 'feature_manifest.csv', index=False)

checks = {
    'complete_oof': complete,
    'outer_validation_labels_used_in_features': False,
    'inner_target_features_cross_fitted': True,
    'test_probabilities_finite_in_01': bool(np.isfinite(test_pred).all() and ((test_pred >= 0) & (test_pred <= 1)).all()),
    'oof_id_order_equals_train': True,
    'test_id_order_equals_test': True,
}
write_json('checks.json', checks)
write_json('config.json', {
    'model': 'L2-regularized logistic regression / binomial GLM',
    'optimizer': 'torch.optim.LBFGS', 'l2': L2, 'lbfgs_iters': LBFGS_ITERS,
    'output_transform': {'temperature': OUT_TEMPERATURE, 'offset': OUT_OFFSET},
    'cv': {'outer_folds': N_FOLDS, 'inner_folds': INNER_FOLDS, 'shuffle': True, 'random_state': RANDOM_STATE},
    'n_features': P, 'device': str(DEVICE),
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                 'torch': torch.__version__, 'sklearn': importlib.metadata.version('scikit-learn')}
})
summary.update(status='complete' if complete else 'partial_smoke', pooled_oof_auc=pooled_auc,
               runtime_seconds=round(time.time() - run_started, 2),
               prediction_files={'oof': str(oof_path), 'test': str(test_path), 'submission': str(SUBMISSION_PATH)})
write_json('run_summary.json', summary)

print('Saved:', oof_path)
print('Saved:', test_path)
print('Saved:', SUBMISSION_PATH)